# POGOS: SOSSSISSE step-by-step

This is the basic steps to run just the SOSSISSE steps using a POGOS yaml 
file.

This assumes you have run `pogos_setup.py` first to generate the file structure and the yaml file mentioned below.

## Imports and notebook setup

In [ ]:
import numpy as np

from sossisse.core import io
from sossisse.core import misc
from sossisse.general import general
from sossisse.general import plots

from pogos.resources import nb_funcs

## Yaml file definition

Set the path to the POGOS yaml file here

In [ ]:
yaml_file = '/scratch3/jwst/pogos-data/soss_test.yaml'

## Initialization

POGOS and SOSSISSE initialization, don't change this section

In [ ]:
# pogos sossisse init
inst = nb_funcs.sossiss_init(yaml_file)
# linear recon initialization
general.linear_recon_init(inst)

## Update constants

Here you can update constants


In [ ]:
# inst.params[PARAM_NAME] = VALUE

## Fancy Centering 

This does the fancy centering
- Maybe remove this?
- Or at least move it later (after background correction etc)

In [ ]:
# fancy centering - recalculate trace file
inst.fancy_centering()

## Load the data and apply DQ


In [ ]:
# -------------------------------------------------------------------------
# load the image, error and data quality
cube, err, dq = inst.load_data_with_dq()
# -------------------------------------------------------------------------
# apply the dq to the cube
cube, err = inst.apply_dq(cube, err, dq)

## Remove background

In [ ]:
# remove the background
cube, err = inst.remove_background(cube, err)

## Patch isolated bad pixels

In [ ]:
# for each slice of the cube, isolated bad pixels are interpolated with the
# value of their 4 neighbours.
cube = inst.patch_isolated_bads(cube)

## Remove cosmic rays

In [ ]:
# remove cosmic rays with a sigma cut
cube = inst.remove_cosmic_rays(cube)

## Get the trace map

In [ ]:
# get the trace map
tracemap = inst.get_trace_map()

## Differential 1/f correction

In [ ]:
cube, med, transit_invsout = inst.clean_1f(cube, err, tracemap)

## PCA Analysis

Note turn this off with 

```
SOSSISSE.WLC.LMODEL.FIT_PCA = False
```

In [ ]:
# construct the principal component model from the out of transit domain
# using pca (we deal with not fitting the PCA inside)
pcas = inst.fit_pca(cube, err, med, tracemap)

## Recenter the trace position

In [ ]:
tracemap = inst.recenter_trace_position(tracemap, med)

## Linear reconstruction

In [ ]:
# get the gradients
dx, dy, rotxy, ddy, med_clean = inst.get_gradients(med)
# set up the mask for trace position
mask_out = inst.get_mask_trace_pos(med, tracemap)
mask_trace_pos, x_order0, y_order0, x_trace_pos, y_trace_pos = mask_out

# setup the linear reconstruction vector based on the input parameters
lvector = inst.setup_linear_reconstruction(med, dx, dy, rotxy, ddy,
                                           pcas)
# -----------------------------------------------------inst--------------------
# find the best linear combination of scale/dx/dy/rotation from lvector
# amps is a vector with the amplitude of all 4 fitted terms
# amps[0] -> amplitude of trace
# amps[1] -> dx normalized on reference trace
# amps[2] -> dy normalized on reference trace
# amps[3] -> rotation (in radians) normalized on reference trace
# amps[4] -> 2nd derivative in y [if option activated]
# -------------------------------------------------------------------------
amp_out = inst.apply_amp_recon(cube, err, med, mask_trace_pos,
                               lvector, x_trace_pos, y_trace_pos,
                               x_order0, y_order0)
# get outputs of apply_amp_recon
ltable, lrecon, valid_cube = amp_out

## Transit part

In [ ]:
# -------------------------------------------------------------------------
# At this point we can look at the transit
inst.define_transit_ints(ltable)
# -------------------------------------------------------------------------
# normalize the trace but a normalization factor
ltable = inst.normalize_sum_trace(ltable)
# -------------------------------------------------------------------------
# per pixel baseline
if inst.params['WLC.GENERAL.PER_PIXEL_BASELINE_CORRECTION']:
    misc.printc('Performing per-pixel baseline subtraction', 'info')
    cube = inst.per_pixel_baseline(cube, valid_cube)
# -------------------------------------------------------------------------
# print the rms baseline for all methods
for method in inst.get_rms_baseline():
    # calculate the rms for this method
    rms_method = inst.get_rms_baseline(ltable['amplitude'], method=method)
    # print this
    msg = '{0}, rms = {1:.1f}ppm'.format(method, rms_method * 1e6)
    misc.printc(msg, 'number')
# -------------------------------------------------------------------------
# get the effective wavelength
photo_weighted_mean, energy_weighted_mean = inst.get_effective_wavelength()

## Save and plot the linear recon results

In [ ]:
# =========================================================================
# write files
# =========================================================================
inst.save_wlc_results(cube, err, lrecon, ltable)
# =========================================================================
# Plots and Summary HTML
# =========================================================================
# plot the stability plot
plots.plot_stability(inst, ltable)
# -------------------------------------------------------------------------
# plot the transit plot
plots.plot_transit(inst, ltable)
# -------------------------------------------------------------------------
# write the yaml file to html
io.summary_html(inst.params)

## Spectral extraction setup

In [ ]:
# get parameters from instrumental parameters
objname = inst.params['INPUTS.OBJECTNAME']
# print the white light curve splash
print(misc.art('Spectral timeseries ' + objname, 'blue', 'CYAN'))
# -------------------------------------------------------------------------
# load temporary filenames (should be run before science starts)
inst.define_filenames()
# -------------------------------------------------------------------------
# plot / save storage
storage = dict()
# get the trace orders
trace_orders = inst.get_trace_orders()

## Processing trace order loop

This is done for each order, you could remove the for loop and just set the
trace order individually
```
trace_order = 1
```

In [ ]:
# loop around trace orders
for trace_order in trace_orders:
    # print progress
    misc.printc('Processing trace order {0}'.format(trace_order), 'alert')
    # ---------------------------------------------------------------------
    # load data for this trace order
    indata = inst.load_input_spec_data(trace_order)
    med, dx, dy, rotxy, ddy, med_clean, residual, err = indata[:8]
    recon, ltable, posmax, throughput, wavegrid = indata[8:]
    # ---------------------------------------------------------------------
    # create the SED
    sp_sed = inst.create_sed(med, residual, wavegrid, posmax, throughput,
                             med_clean, trace_order)
    # ---------------------------------------------------------------------
    # load the model (and deal with masking order zero if required)
    model = inst.load_model(recon, med)
    # ---------------------------------------------------------------------
    # spectrum is the ratio of the residual to the trace model
    spec, spec_err = inst.ratio_residual_to_trace(model, err, residual,
                                                  posmax)
    # ---------------------------------------------------------------------
    # remove the out-of-transit trend on the spectrum
    if inst.params['SPEC_EXT.REMOVE_TREND']:
        spec = inst.remove_trend_spec(spec)
    # -----------------------------------------------------------------
    # reshape the amplitudes into an image
    amp_image = np.repeat(np.array(ltable['amplitude']), spec.shape[1])
    amp_image = amp_image.reshape(spec.shape)
    # add this gray component onto the spectrum
    spec2 = spec + amp_image
    # ---------------------------------------------------------------------
    # remove the out-of-transit trend on the photometric time series
    if inst.params['SPEC_EXT.REMOVE_TREND']:
        ltable = inst.remove_trend_phot(spec, ltable)
    # ---------------------------------------------------------------------
    # compute or set transit depth
    transit_depth = inst.get_transit_depth(ltable)
    # ---------------------------------------------------------------------
    # get the in-transit spectrum
    isout = inst.intransit_spectrum(spec, spec_err)
    spec_in, spec_err_in, spec_err_out = isout
    # ---------------------------------------------------------------------
    # bin the data by RESOLUTION_BIN
    wave_bin, flux_bin, flux_bin_err = inst.bin_spectrum(wavegrid,
                                                         spec_in,
                                                         spec_err_in)
    # ---------------------------------------------------------------------
    # push into storage for outside loop
    storage = general.trace_storage(storage, trace_order, wavegrid, sp_sed,
                                    throughput, spec, spec_err, ltable, spec2,
                                    spec_in, spec_err_in, transit_depth, 
                                    wave_bin, flux_bin, flux_bin_err)
    # ---------------------------------------------------------------------
    inst.save_spe_results(storage[trace_order], trace_order)

## Spectral extraction plots and saving

In [ ]:
# plot the SED
plots.plot_full_sed(inst, storage)
# -------------------------------------------------------------------------
# convert sossisse to eureka products
inst.to_eureka(storage)
# -------------------------------------------------------------------------
# write the yaml file to html
io.summary_html(inst.params)